# Fase 3.4 v2 — amostragem com raiz quadrada
Reutiliza os dados persistidos da v1 e o checkpoint da Fase 3.3. Execute 1–6 para preparar a sessão, depois 7 para smoke test. A célula 8 inicia ou retoma o treino somente quando executada manualmente. A célula 9 lê os artefatos da v2. Nenhuma célula treina automaticamente ao abrir o notebook.

In [12]:
# 1. GPU e dependências
import torch
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab'
print('GPU:', torch.cuda.get_device_name(0))
print('Verificando/instalando dependências...')
%pip install datasets scikit-learn cryptography

GPU: Tesla T4
Verificando/instalando dependências...


In [13]:
# 2. Drive e pastas persistentes; nenhuma pasta de execução anterior é alterada.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for folder in ('credenciais', 'modelos', 'dados_fase3_4', 'experimentos'):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final


In [14]:

# 3. Desbloqueio da credencial GitHub
# Compatível com os formatos antigo e novo de github.enc

import base64
import getpass
import json

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

credential_path = ROOT / "credenciais" / "github.enc"

if not credential_path.is_file():
    raise FileNotFoundError(
        f"Credencial não encontrada: {credential_path}"
    )

credential = json.loads(
    credential_path.read_text(encoding="utf-8")
)

# Identifique pelo nome do campo cifrado, sem exigir metadados no formato antigo.
if not isinstance(credential, dict) or "salt" not in credential:
    raise ValueError("Formato de github.enc não reconhecido.")
if "ciphertext" in credential and "token" in credential:
    raise ValueError("Credencial ambígua: dois campos cifrados.")
if "ciphertext" in credential:
    if credential.get("version") != 1:
        raise ValueError("Versão de credencial não suportada.")
    iterations = int(credential["iterations"])
    encrypted_token = credential["ciphertext"]
elif "token" in credential:
    iterations = 600000
    encrypted_token = credential["token"]

else:
    raise ValueError(
        "Formato de github.enc não reconhecido."
    )
if iterations < 600000 or not isinstance(encrypted_token, str):
    raise ValueError("Parâmetros da credencial inválidos.")

salt = base64.b64decode(credential["salt"])

password = getpass.getpass(
    "Senha de desbloqueio do GitHub: "
)

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=iterations
)

key = base64.urlsafe_b64encode(
    kdf.derive(password.encode())
)

try:
    token = Fernet(key).decrypt(
        encrypted_token.encode()
    ).decode()

except InvalidToken:
    raise ValueError(
        "Senha incorreta ou credencial corrompida."
    ) from None

finally:
    del password, kdf, key

print("Credencial desbloqueada nesta sessão.")


Credencial desbloqueada nesta sessão.


In [15]:
# 4. Snapshot imutável da branch privada no runtime; não faz git clone nem toca no Drive.
import io, os, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
OWNER, REPO, BRANCH = 'vicemnovo', 'TCC_Final', 'fase3-ajuste-fino'
API = f'https://api.github.com/repos/vicemnovo/TCC_Final'
def github_read(url):
    request = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase3-Colab'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            chunks, size, last_notice = [], 0, time.monotonic()
            while chunk := response.read(1024 * 1024):
                chunks.append(chunk)
                size += len(chunk)
                if time.monotonic() - last_notice >= 30:
                    print(f'Download GitHub em andamento: {size / 1048576:.1f} MiB', flush=True)
                    last_notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code} ao consultar {url.split(API)[-1] or "/"}') from None
try:
    repo_info = json.loads(github_read(API))
except RuntimeError as error:
    raise RuntimeError('Repositório privado inacessível: verifique owner, seleção do repositório no token e permissão Metadata (read).') from error
try:
    branch_info = json.loads(github_read(f'{API}/branches/{BRANCH}'))
except RuntimeError as error:
    if 'HTTP 404' not in str(error):
        raise
    raise RuntimeError(f'Repositório acessível, mas a branch {BRANCH!r} não foi encontrada. Publique essa branch; a branch padrão é {repo_info["default_branch"]!r}.') from error
commit = branch_info['commit']['sha']
assert re.fullmatch(r'[0-9a-f]{40}', commit)
CODE_ROOT = Path('/content/tcc_codigo') / commit
required = ('experimentos/fase3_4_finetune_v2.py', 'experimentos/amostragem_fase3_4.py', 'experimentos/particoes_fase3_4.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
if not CODE_ROOT.exists():
    print('Baixando snapshot do código...', flush=True)
    archive = github_read(f'{API}/zipball/{commit}')
    CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='tcc_sync_', dir=CODE_ROOT.parent) as temporary:
        staged = Path(temporary)
        print('Extraindo snapshot no runtime...', flush=True)
        with zipfile.ZipFile(io.BytesIO(archive)) as zf:
            last_notice = time.monotonic()
            extracted = 0
            for item in zf.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or len(parts) < 2:
                    continue
                relative = Path(*parts[1:])
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Caminho inseguro no snapshot')
                destination = staged / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(item) as source, destination.open('wb') as output:
                    shutil.copyfileobj(source, output)
                extracted += 1
                if time.monotonic() - last_notice >= 30:
                    print(f'Extração em andamento: {extracted} arquivos', flush=True)
                    last_notice = time.monotonic()
        if not all((staged / name).is_file() for name in required):
            raise FileNotFoundError('Publique a branch com o script e o notebook original antes de sincronizar')
        staged.rename(CODE_ROOT)
assert all((CODE_ROOT / name).is_file() for name in required)
del token
print('Código sincronizado:', commit)
print('Pasta local do runtime:', CODE_ROOT)

Baixando snapshot do código...
Extraindo snapshot no runtime...
Código sincronizado: 37fc72c3b9482fb8d1ed8f51f57a6964245029e5
Pasta local do runtime: /content/tcc_codigo/37fc72c3b9482fb8d1ed8f51f57a6964245029e5


In [16]:
# 5. Versão, dependências e checkpoint inicial, sem treino.
import importlib, subprocess, sys
for module in ('torch', 'torchvision', 'datasets', 'sklearn', 'PIL', 'numpy', 'cryptography'):
    importlib.import_module(module)
SCRIPT = CODE_ROOT / 'experimentos/fase3_4_finetune_v2.py'
MODULE = CODE_ROOT / 'experimentos/amostragem_fase3_4.py'
SPLITS_MODULE = CODE_ROOT / 'experimentos/particoes_fase3_4.py'
SOURCE = CODE_ROOT / 'Notebooks/fase_3/fase3.3_compressao.ipynb'
assert SCRIPT.is_file() and MODULE.is_file() and SPLITS_MODULE.is_file() and SOURCE.is_file()
for relative in ('modelos/fase2_gate_vit_fft_cnn/phase2_final.pt', 'modelos/fase3_3_compressao_v1/phase3_final.pt'):
    assert (ROOT / relative).is_file(), f'Checkpoint ausente: {ROOT / relative}'
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT), str(MODULE), str(SPLITS_MODULE)], check=True)
ck = torch.load(ROOT / 'modelos/fase3_3_compressao_v1/phase3_final.pt', map_location='cpu', weights_only=False)
assert 'model_state_dict' in ck
del ck
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
ENV.pop('HF_TOKEN', None)  # a v2 usa apenas os dados já preparados
print('Checkpoint e script verificados; commit:', commit)

# Executa o script com logs em tempo real e aviso periódico de atividade.
import queue, threading, time
def executar_com_progresso(modo, intervalo=30):
    ambiente = ENV.copy()
    ambiente['PYTHONUNBUFFERED'] = '1'
    processo = subprocess.Popen([sys.executable, '-u', str(SCRIPT), modo], env=ambiente,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
    mensagens = queue.Queue()
    def ler_saida():
        for linha in processo.stdout:
            mensagens.put(linha)
        mensagens.put(None)
    threading.Thread(target=ler_saida, daemon=True).start()
    inicio, terminou = time.monotonic(), False
    print(f'{modo}: processo iniciado (PID {processo.pid})', flush=True)
    while not terminou:
        try:
            linha = mensagens.get(timeout=intervalo)
            if linha is None:
                terminou = True
            else:
                print(linha, end='', flush=True)
        except queue.Empty:
            print(f'{modo}: em execução há {(time.monotonic() - inicio) / 60:.1f} min; aguardando próxima saída...', flush=True)
    codigo = processo.wait()
    if codigo:
        raise RuntimeError(f'{modo} falhou (código {codigo}); veja o traceback acima')
    print(f'{modo}: concluído em {(time.monotonic() - inicio) / 60:.1f} min', flush=True)

Checkpoint e script verificados; commit: 37fc72c3b9482fb8d1ed8f51f57a6964245029e5


In [8]:
# 6. Verificação dos dados persistidos da v1; a v2 não baixa nem modifica o manifesto.
manifest = ROOT / 'dados_fase3_4/manifest.json'
if not manifest.is_file():
    raise FileNotFoundError(f'Manifesto da v1 ausente: {manifest}')
else:
    print('Dados existentes preservados:', manifest)

Dados existentes preservados: /content/drive/MyDrive/TCC_Final/dados_fase3_4/manifest.json


In [9]:
# 7. Smoke test sem treino; não repete se já houver resultado.
smoke = ROOT / 'modelos/fase3_4_finetune_v2_sqrt_sampler_smoke/smoke.json'
if not smoke.is_file():
    executar_com_progresso('smoke')
else:
    print('Smoke anterior preservado:', smoke)

smoke: processo iniciado (PID 2592)
GPU: Tesla T4
Fase 2 carregada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições v2: treino= 1668 | seleção= 182 | calibração= 182 | teste reservado= 1947 | replay= 0
smoke: em execução há 0.8 min; aguardando próxima saída...
smoke: em execução há 1.3 min; aguardando próxima saída...
smoke: em execução há 1.8 min; aguardando próxima saída...
smoke: em execução há 2.3 min; aguardando próxima saída...
smoke: em execução há 2.8 min; aguardando próxima saída...
smoke: em execução há 3.3 min; aguardando próxima saída...
smoke: em execução há 3.8 min; aguardando próxima saída...
smoke: em execução há 4.3 min; aguardando próxima saída...
smoke: em execução há 4.8 min; aguardando próxima saída...
smoke: em execução há 5.3 min; aguardando próxima saída...
smoke: em execução há 5.8 min; aguardando próxima saída...
smoke: em execução há 6.3 min; aguardando próxima saída...
smoke: em execução há 6.8 min; aguar

In [10]:
# 8. EXECUÇÃO MANUAL: inicia ou retoma conforme last.pt; não rode até querer treinar.
executar_com_progresso('auto')

auto: processo iniciado (PID 9082)
GPU: Tesla T4
Fase 2 carregada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições v2: treino= 1668 | seleção= 182 | calibração= 182 | teste reservado= 1947 | replay= 0
Sampler v2: classes 50/50; massa por grupo proporcional a sqrt(n).
auto: em execução há 0.7 min; aguardando próxima saída...
[E01/12] lr=3.0e-05 | train_loss=0.4848 | val_loss=0.3227 | AUC=0.9591 | bal_acc=0.8945 | 35s  -> melhor
[E02/12] lr=3.0e-05 | train_loss=0.4647 | val_loss=0.3128 | AUC=0.9640 | bal_acc=0.9134 | 10s  -> melhor
[E03/12] lr=2.9e-05 | train_loss=0.4698 | val_loss=0.3268 | AUC=0.9610 | bal_acc=0.8884 | 11s
[E04/12] lr=2.8e-05 | train_loss=0.4523 | val_loss=0.3090 | AUC=0.9635 | bal_acc=0.8923 | 10s
[E05/12] lr=2.5e-05 | train_loss=0.4611 | val_loss=0.3093 | AUC=0.9657 | bal_acc=0.9145 | 10s  -> melhor
[E06/12] lr=2.1e-05 | train_loss=0.4535 | val_loss=0.3030 | AUC=0.9671 | bal_acc=0.9084 | 12s  -> melhor
[E07/12] lr=

In [11]:
# 9. Conferência somente-leitura dos artefatos; não reavalia nem altera o teste.
import csv
run_dir = ROOT / 'modelos/fase3_4_finetune_v2_sqrt_sampler'
artifacts = ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt', 'best.pt', 'phase3_final.pt', 'final_metrics.json', 'per_group.csv', 'test_predictions.csv')
for name in artifacts:
    path = run_dir / name
    print(f'{name}: ' + (f'presente ({path.stat().st_size:,} bytes)' if path.is_file() else 'AUSENTE'))
history = run_dir / 'history.csv'
if history.is_file():
    with history.open(newline='', encoding='utf-8') as handle:
        epochs = list(csv.DictReader(handle))
    print('Última época no histórico:', epochs[-1]['epoch'] if epochs else 'histórico vazio')
result = run_dir / 'final_metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2) if result.is_file() else 'Ainda sem resultados finais.')

experiment_config.json: presente (1,519 bytes)
split_manifest.json: presente (611,057 bytes)
history.csv: presente (1,273 bytes)
last.pt: presente (121,448,069 bytes)
best.pt: presente (40,624,489 bytes)
phase3_final.pt: presente (40,628,318 bytes)
final_metrics.json: presente (514 bytes)
per_group.csv: presente (1,665 bytes)
test_predictions.csv: presente (220,649 bytes)
Última época no histórico: 9
{
  "accuracy": 0.7288135593220338,
  "precision": 0.7804551539491299,
  "recall": 0.6156282998944034,
  "f1": 0.6883116883116883,
  "tnr": 0.836,
  "bal_acc": 0.7258141499472017,
  "tn": 836,
  "fp": 164,
  "fn": 364,
  "tp": 583,
  "auc": 0.8222402323125659,
  "threshold": 0.36999999999999994,
  "temperature": 0.44025686383247375,
  "validation_calibration_bal_acc": 0.9079268292682927,
  "best_epoch": 6,
  "start_checkpoint": "/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt"
}


In [17]:
# 10. AVALIAÇÃO EXTERNA FINAL, MANUAL E ÚNICA; não treina nem calibra.
# Execute somente após publicar o script novo na branch e sincronizar as células 1–5.
EXTERNAL_SCRIPT = CODE_ROOT / 'experimentos/avaliar_externo_fase3_4_v2.py'
assert EXTERNAL_SCRIPT.is_file(), 'Publique o script de avaliação externa antes de executar.'
external_dir = ROOT / 'resultados/avaliacao_externa_fase3_4_v2_v1'
external_metrics = external_dir / 'metrics.json'
if external_dir.exists():
    if not external_metrics.is_file():
        raise RuntimeError(f'Avaliação anterior incompleta; preserve e investigue: {external_dir}')
    print('Avaliação externa anterior preservada:', external_dir)
    print(json.dumps(json.loads(external_metrics.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('Iniciando avaliação externa congelada; a conferência de hashes pode demorar.', flush=True)
    proc = subprocess.Popen([sys.executable, '-u', str(EXTERNAL_SCRIPT), '--root', str(ROOT), '--code-root', str(CODE_ROOT)],
                            env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'Avaliação externa falhou (código {proc.returncode}); veja a saída acima')
    assert external_metrics.is_file(), 'Processo terminou sem metrics.json'

Iniciando avaliação externa congelada; a conferência de hashes pode demorar.
Conferindo sobreposição exata com 3979 imagens OpenFake...
Integridade: 100/3979 arquivos verificados (0.0 min)
Integridade: 200/3979 arquivos verificados (0.0 min)
Integridade: 300/3979 arquivos verificados (0.0 min)
Integridade: 400/3979 arquivos verificados (0.0 min)
Integridade: 500/3979 arquivos verificados (0.0 min)
Integridade: 600/3979 arquivos verificados (0.0 min)
Integridade: 700/3979 arquivos verificados (0.0 min)
Integridade: 800/3979 arquivos verificados (0.0 min)
Integridade: 900/3979 arquivos verificados (0.0 min)
Integridade: 1000/3979 arquivos verificados (0.0 min)
Integridade: 1100/3979 arquivos verificados (0.0 min)
Integridade: 1200/3979 arquivos verificados (0.0 min)
Integridade: 1300/3979 arquivos verificados (0.0 min)
Integridade: 1400/3979 arquivos verificados (0.0 min)
Integridade: 1500/3979 arquivos verificados (0.0 min)
Integridade: 1600/3979 arquivos verificados (0.0 min)
Integrida